# Colab Full Run (Food101 + STL10)

This notebook clones the repo, installs deps, downloads Food101, runs extraction/training/eval/autointerp, and builds tables.


In [ ]:
# Repo clone
REPO_URL = "https://github.com/<USER>/<REPO>.git"  # TODO: set
REPO_DIR = "sae_for_clip"

import os

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}


In [ ]:
# Install deps
!pip install -r sae_for_clip/requirements.txt
!pip install -e sae_for_clip


In [ ]:
# Download Food101 via torchvision
from torchvision.datasets import Food101
Food101(root="sae_for_clip/data/torchvision", split="train", download=True)
FOOD_IMAGES = "sae_for_clip/data/torchvision/food-101/images"


In [ ]:
# Extract activations
RUN_ACT = "colab_food101_acts"
!PYTHONPATH=sae_for_clip/src python sae_for_clip/scripts/extract_activations.py \
  --image_dir {FOOD_IMAGES} \
  --model ViT-B-32 \
  --pretrained openai \
  --layer visual.transformer.resblocks.0 \
  --num_samples 50000 \
  --batch_size 128 \
  --shard_size 4096 \
  --out_dir sae_for_clip/artifacts/activation_cache \
  --run_name {RUN_ACT}


In [ ]:
# Train MSAE
RUN_SAE = "colab_msae"
!PYTHONPATH=sae_for_clip/src python sae_for_clip/scripts/train_sae.py \
  --sae_type msae \
  --k_list 32,64,128,256 \
  --alpha_mode reverse \
  --input_centering dataset \
  --input_scaling dataset \
  --cache_dir sae_for_clip/artifacts/activation_cache/colab_food101_acts \
  --dict_size 8192 \
  --epochs 10 \
  --batch_size 1024 \
  --lr 1e-3 \
  --l1_lambda 1e-4 \
  --device cuda \
  --run_name {RUN_SAE}


In [ ]:
# Zero-shot eval (CIFAR-10 + STL-10)
!PYTHONPATH=sae_for_clip/src python sae_for_clip/scripts/eval_zeroshot.py --dataset cifar10 --split test --batch_size 256 --num_workers 2 --run_name c10_base
!PYTHONPATH=sae_for_clip/src python sae_for_clip/scripts/eval_zeroshot.py --dataset stl10 --split test --batch_size 256 --num_workers 2 --run_name stl10_base

!PYTHONPATH=sae_for_clip/src python sae_for_clip/scripts/eval_zeroshot.py \
  --dataset cifar10 --split test --batch_size 256 --num_workers 2 \
  --sae_checkpoint sae_for_clip/artifacts/checkpoints/colab_msae/last.pt \
  --sae_layer visual.transformer.resblocks.0 \
  --run_name c10_sae

!PYTHONPATH=sae_for_clip/src python sae_for_clip/scripts/eval_zeroshot.py \
  --dataset stl10 --split test --batch_size 256 --num_workers 2 \
  --sae_checkpoint sae_for_clip/artifacts/checkpoints/colab_msae/last.pt \
  --sae_layer visual.transformer.resblocks.0 \
  --run_name stl10_sae

!PYTHONPATH=sae_for_clip/src python sae_for_clip/scripts/make_p4_table.py \
  --eval_runs sae_for_clip/artifacts/eval/c10_sae sae_for_clip/artifacts/eval/stl10_sae \
  --checkpoint_dir sae_for_clip/artifacts/checkpoints/colab_msae \
  --out_path sae_for_clip/artifacts/eval/zeroshot_eval_table.md


In [ ]:
# Auto-interpretation (collages + CSV + table)
!PYTHONPATH=sae_for_clip/src python sae_for_clip/scripts/build_collages.py \
  --cache_dir sae_for_clip/artifacts/activation_cache/colab_food101_acts \
  --checkpoint sae_for_clip/artifacts/checkpoints/colab_msae/last.pt \
  --num_latents 300 --top_k 16 --image_size 128 \
  --out_dir sae_for_clip/artifacts/autointerp/collages \
  --manifest_path sae_for_clip/artifacts/autointerp/collages_manifest.csv

from getpass import getpass
import os
os.environ['OPENROUTER_API_KEY'] = getpass('OpenRouter API key: ')

!PYTHONPATH=sae_for_clip/src python sae_for_clip/scripts/autointerp.py \
  --manifest_path sae_for_clip/artifacts/autointerp/collages_manifest.csv \
  --out_csv sae_for_clip/artifacts/autointerp/autointerp.csv

!PYTHONPATH=sae_for_clip/src python sae_for_clip/scripts/make_p5_table.py \
  --autointerp_csv sae_for_clip/artifacts/autointerp/autointerp.csv \
  --out_path sae_for_clip/artifacts/autointerp/autointerp_table.md
